In [1]:
DATA  = "/kaggle/input/datasets/karrychuong/qudar-beir-data/data"
PATCH = "/kaggle/input/datasets/karrychuong/qudar-patches/qudar-patches"
EXP   = "/kaggle/working/exp"
!mkdir -p {EXP} && cp /kaggle/input/datasets/karrychuong/qudar-expansions/*/expansions/expanded_queries_*.json {EXP}/
!ls {EXP}
!git clone -q https://github.com/kaist-dmlab/QuDAR.git /kaggle/working/QuDAR
!pip install -q -e /kaggle/working/QuDAR bm25s
!python {PATCH}/apply_patch.py /kaggle/working/QuDAR
RUN = f"python {PATCH}/run_retrieval.py --repo /kaggle/working/QuDAR --data-root {DATA} --expansions {EXP} --out /kaggle/working/runs"

expanded_queries_climate-fever_test_expansion.json
expanded_queries_fiqa_test_expansion.json
expanded_queries_fiqa_train_expansion.json
expanded_queries_hotpotqa_test_expansion.json
expanded_queries_hotpotqa_train_expansion.json
expanded_queries_scidocs_test_expansion.json
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Building editable for qudar (pyproject.toml) ... done
patched /kaggle/working/QuDAR/src/qudar/retrievers.py -> BM25sSparseRetriever


In [2]:
!grep -c backend_selection {PATCH}/bm25_fast.py

2


In [3]:
import glob
IDX = glob.glob("/kaggle/input/**/fiqa/contriever/index.faiss", recursive=True)[0].split("/fiqa/contriever")[0]
print("IDX =", IDX)

IDX = /kaggle/input/datasets/karrychuong/qudar-index-small/indexes


In [4]:
!{RUN} --dataset fiqa --splits test train --dense-faiss {IDX}/fiqa/contriever

[19:04:35] fiqa/test: 648 queries
[19:04:35] fiqa/train: 5500 queries
[19:04:35] BM25: reading corpus
[19:04:36] BM25: indexing 57638 documents
[19:04:59] BM25 search test: OS + ES
[19:05:01] BM25 search train: OS + ES
tokenizer_config.json: 100%|███████████████████| 321/321 [00:00<00:00, 1.41MB/s]
vocab.txt: 100%|█████████████████████████████| 232k/232k [00:00<00:00, 13.4MB/s]
tokenizer.json: 100%|████████████████████████| 466k/466k [00:00<00:00, 35.5MB/s]
special_tokens_map.json: 100%|██████████████████| 112/112 [00:00<00:00, 731kB/s]

pytorch_model.bin: downloading bytes: █████████▋            |  194MB, 13.6MB/s  
pytorch_model.bin: downloading bytes: █████████████▏        |  262MB, 22.1MB/s  
pytorch_model.bin: downloading bytes: ████████████████████▋ |  412MB, 34.3MB/s  
pytorch_model.bin: downloading bytes: ██████████████████████|  413MB, 34.4MB/s  
pytorch_model.bin: reconstructing file: 100%|██████|  438MB /  438MB, 38.1MB/s  
Loading weights: 100%|█████████████████████| 199/19

In [5]:
!{RUN} --dataset scidocs --splits test --dense-faiss {IDX}/scidocs/contriever

[19:15:30] scidocs/test: 1000 queries
[19:15:30] BM25: reading corpus
[19:15:32] BM25: indexing 25657 documents
[19:15:38] BM25 search test: OS + ES
Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 14563.81it/s]
[19:15:50] Dense test: encode queries
Dense retrieval: 100%|████████████████████████████| 1/1 [00:00<00:00,  2.08it/s]
[19:16:10] Evaluate test -> /kaggle/working/runs/scidocs_test
Dynamic hybrid sweep: 100%|██████████████| 1000/1000 [00:00<00:00, 60786.13it/s]
            method  Recall@10  nDCG@10  Precision@1
                os     0.1653   0.1578       0.1830
                od     0.0950   0.0889       0.1020
                es     0.1459   0.1323       0.1320
                ed     0.1361   0.1283       0.1460
  qudar_simple_rrf     0.1877   0.1736       0.1860
qudar_simple_equal     0.1901   0.1793       0.2030
  qudar_confidence     0.1907   0.1798       0.2040
         sha_os_od     0.1511   0.1430       0.1635
         shb_os_od     0.1755   0.1664  

In [6]:
import sys, json
from pathlib import Path
sys.path.insert(0, "/kaggle/working/QuDAR/src"); sys.path.insert(0, PATCH)
from run_retrieval import load_split
from qudar.retrievers import build_dense_encoder
from qudar.index import DenseIndexer
from qudar.evaluation import evaluate_run

qids, qtexts, qrels = load_split(Path(DATA) / "fiqa", "test", None)
exp = json.load(open(f"{EXP}/expanded_queries_fiqa_test_expansion.json"))
etexts = [exp[q] for q in qids]
enc = build_dense_encoder("facebook/contriever")
idx = DenseIndexer(768); idx.deserialize_from(f"{IDX}/fiqa/contriever")

def ev(vecs):
    res = idx.search(vecs, 1000)
    return evaluate_run(qrels, {q: {d: float(s) for d, s in zip(*r)} for q, r in zip(qids, res)}, 10)[0]

both = [q + " " + e for q, e in zip(qtexts, etexts)]
for name, vecs in [("A passage: + mở rộng (hiện tại)", enc.encode_corpus(etexts)),
                   ("B query: + mở rộng",             enc.encode_queries(etexts)),
                   ("C passage: + câu hỏi + mở rộng", enc.encode_corpus(both)),
                   ("D query: + câu hỏi + mở rộng",   enc.encode_queries(both))]:
    r = ev(vecs); print(f"{name:34s} R@10 {r['Recall@10']}  nDCG {r['nDCG@10']}  P@1 {r['Precision@1']}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.17it/s]


A passage: + mở rộng (hiện tại)    R@10 0.1261  nDCG 0.0939  P@1 0.0741


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.35it/s]


B query: + mở rộng                 R@10 0.1753  nDCG 0.1198  P@1 0.0772


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.19it/s]


C passage: + câu hỏi + mở rộng     R@10 0.1512  nDCG 0.118  P@1 0.1049


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.26it/s]


D query: + câu hỏi + mở rộng       R@10 0.2036  nDCG 0.1532  P@1 0.1281


In [7]:
qids, qtexts, qrels = load_split(Path(DATA) / "scidocs", "test", None)
exp = json.load(open(f"{EXP}/expanded_queries_scidocs_test_expansion.json"))
etexts = [exp[q] for q in qids]
idx = DenseIndexer(768); idx.deserialize_from(f"{IDX}/scidocs/contriever")
both = [q + " " + e for q, e in zip(qtexts, etexts)]
for name, vecs in [("A", enc.encode_corpus(etexts)), ("B", enc.encode_queries(etexts)),
                   ("C", enc.encode_corpus(both)),   ("D", enc.encode_queries(both))]:
    r = ev(vecs); print(name, r['Recall@10'], r['nDCG@10'], r['Precision@1'])

Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.11it/s]


A 0.1361 0.1283 0.146


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


B 0.1496 0.1428 0.171


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


C 0.1403 0.1333 0.154


Dense retrieval: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]


D 0.1573 0.1469 0.164
